🔹 STEP 1: Load data

In [40]:
import pandas as pd
import numpy as np

data = {
    "CGPA": [8.5,7.0,6.5,9.0,5.5,7.8,6.9],
    "IQ": [120,110,105,130,95,118,112],
    "Height": [170,168,169,171,168,170,169],
    "Experience": [1,2,1,3,0,2,1],
    "Package": [4.2,3.2,2.9,4.8,2.1,3.9,3.4]
}

df = pd.DataFrame(data)

X = df.drop("Package", axis=1)
y = df["Package"]


🔹 STEP 2: Train–Test Split ✅ (IMPORTANT)

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


🔹 STEP 3: Lasso for Feature Selection (on TRAIN data)

In [42]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)   # reuse scaler

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)

for feature, coef in zip(X.columns, lasso.coef_):
    print(feature, ":", coef)


CGPA : 0.05609659033016386
IQ : 0.7548784052015158
Height : 0.0
Experience : 0.0


🔹 STEP 4: Select Important Features

In [43]:
selected_features = X.columns[lasso.coef_ != 0]
print("Selected features:", selected_features)

X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]


Selected features: Index(['CGPA', 'IQ'], dtype='object')


🔹 STEP 5A: Test Multiple Linear Regression ✅

In [44]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

mlr = LinearRegression()
mlr.fit(X_train_selected, y_train)

y_pred = mlr.predict(X_test_selected)

print("MLR R2:", r2_score(y_test, y_pred)*100)
print("MLR RMSE:",np.sqrt(mean_squared_error(y_test, y_pred)))


MLR R2: 96.27173821324216
MLR RMSE: 0.09654353663966629


🔹 STEP 5B: Test Polynomial Linear Regression ✅

In [45]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train_selected)
X_test_poly = poly.transform(X_test_selected)

poly_lr = LinearRegression()
poly_lr.fit(X_train_poly, y_train)

y_pred_poly = poly_lr.predict(X_test_poly)

print("Polynomial R2:", r2_score(y_test, y_pred_poly)*100)
print("Polynomial RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_poly)))


Polynomial R2: 96.8510563491341
Polynomial RMSE: 0.0887263158660651
